# Constitutional RAG

## Prerequisits

In [1]:
# imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import pymupdf
import os
import re 

In [2]:
# --- 1. The "Bulletproof" PDF Cleaning Function ---
def clean_pdf_text(text: str) -> str:
    # 1. Replace tabs with spaces
    text = text.replace('\t', ' ')
    
    # 2. Fix words broken by hyphens at line breaks (e.g., "con-\nstitution")
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    
    # 3. Remove isolated page numbers (lines containing ONLY a number)
    text = re.sub(r'^\s*\d{1,4}\s*$', '', text, flags=re.MULTILINE)
    
    # 4. Remove running headers/footers (e.g., "Chapter 1: Founding Provisions")
    text = re.sub(r'(?m)^\s*Chapter \d+:\s+[A-Za-z ]+\s*$', '', text)
    
    # 5. THE MAGIC TRICK: Fix PDF line-wraps without losing paragraph breaks
    # PDFs put a single \n at the end of every visual line. We want to turn those into spaces,
    # but keep the double \n\n that separate actual paragraphs.
    text = text.replace('\n\n', '[[PARA]]')  # Temporarily hide real paragraphs
    text = text.replace('\n', ' ')           # Turn visual line-breaks into spaces
    text = text.replace('[[PARA]]', '\n\n')  # Restore real paragraphs
    
    # 6. Collapse multiple spaces into a single space
    text = re.sub(r' +', ' ', text)
    
    return text.strip()

In [ ]:
# --- 1.  File path
file_path = "../data/RSAConstitution-eng.pdf"

# --- 2. Extract Text using PyMuPDF ---
doc = pymupdf.open(file_path)
raw_text = ""
for page in doc:
    raw_text += page.get_text("text") + "\n"


# --- 3. Strip the "Junk" (Table of Contents & Title Page) ---
preamble_marker = "PREAMBLE\nWe, the people of South Africa"
preamble_index = raw_text.find(preamble_marker)

if preamble_index != -1:
    raw_text = raw_text[preamble_index:]

# --- NEW: Strip the Index (appears at the end) ---
# Look for "INDEX" as a standalone heading
index_marker = "\nINDEX\n"
index_position = raw_text.find(index_marker)

if index_position != -1:
    raw_text = raw_text[:index_position]
    print(f"✓ Removed Index starting at position: {index_position}")
else:
    # Try alternative formats
    alt_markers = ["\nI N D E X\n", "\nINDEX\n\n", "INDEX\n"]
    for marker in alt_markers:
        index_position = raw_text.find(marker)
        if index_position != -1:
            raw_text = raw_text[:index_position]
            print(f"✓ Removed Index using alternative marker")
            break

# Clean the text
text = clean_pdf_text(raw_text)


# --- 4. Wrap the text in a LangChain Document ---
source_document = Document(
    page_content=text, 
    metadata={
        "source": "RSA Constitution 1996", 
        "document_type": "legal"
    }
)


# --- 5. Configure the Text Splitter (Simplified to prevent micro-chunking) ---
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=[
        r"\n\n(?=\d+\.\s+[A-Z])",  # 1. Main sections (e.g., "9. Equality")
        "\n\n",                   # 2. Paragraph breaks
        " ",                      # 3. Spaces (prevents breaking mid-sentence)
        ""                        # 4. Characters (last resort)
    ],
    is_separator_regex=True,
)

# --- 6. Split and Filter ---
chunks = text_splitter.split_documents([source_document])

# print(chunks)
print(f"The number of chuncks {len(chunks)}")

# Let's look at the actual structure of the first chunk
first_chunk = chunks[0]

print("\n--- FIRST CHUNK STRUCTURE ---")
print(f"Text Preview: {first_chunk.page_content[:150]}...")
print(f"Metadata Attached: {first_chunk.metadata}")


✓ Removed Index starting at position: 306535
The number of chuncks 403

--- FIRST CHUNK STRUCTURE ---
Text Preview: PREAMBLE We, the people of South Africa, Recognise the injustices of our past; Honour those who suffered for justice and freedom in our land; Respect ...
Metadata Attached: {'source': 'RSA Constitution 1996', 'document_type': 'legal'}
